In [2]:
from satpy import Scene
import pandas as pd
import ast
import s3fs
from tqdm import tqdm

# Connect to AWS public buckets
fs = s3fs.S3FileSystem(anon=True)

In [3]:
def compile_tcprimed_files(years=[2023, 2024], basins=['AL', 'EP', 'IO', 'SH', 'WP'], product='final'):
    """
    Checks AWS S3 for TC-Primed files and compiles a DataFrame with the file paths and metadata.
    """
    df = pd.DataFrame(columns=['year', 'basin', 'storm_number', 'storm_id', 'satellite', 'time', 'size', 'file_path'])

    for year in tqdm(years, desc="Processing years"):
        for basin in tqdm(basins, desc=f"Processing basins for year {year}"):
            # Construct the S3 path for the TC-Primed files
            s3_base_path = f's3://noaa-nesdis-tcprimed-pds/v01r01/{product}/{year}/{basin}'
            
            try:
                folders = fs.ls(s3_base_path)
                for folder in folders:
                    number = folder.split('/')[-1]  # Extract the storm number from the folder name
                    files = fs.ls(folder)
                    for file in files:
                        if file.endswith('.nc'):
                            # Extract metadata from the file name
                            file_name = file.split('/')[-1]
                            file_name = file_name.split('.')[0]  # Remove the .nc extension
                            parts = file_name.split('_') # TCPRIMED_v01r01-final_AL122023_AMSR2_GCOMW1_060020_20230830024032
                            if len(parts) == 7:
                                _, _, storm_name, satellite, _, _, time  = parts
                            elif len(parts) == 6:
                                _, _, storm_name, satellite, time, _  = parts
                                time = time[1:] # Remove the leading 's' from the time string

                            size = fs.info(file)['Size'] / (1024 ** 3) # Convert to GB
                                
                            df_tmp = pd.DataFrame({
                                'year': [year],
                                'basin': [basin],
                                'storm_number': [number],
                                'storm_id': [storm_name],
                                'satellite': [satellite],
                                'time': [time],
                                'size': [size],
                                'file_path': [file]
                            })
                            df = pd.concat([df, df_tmp], ignore_index=True)
            except Exception as e:
                print(f"Error accessing {s3_base_path}: {e}")
                continue

    return df
                


In [4]:
df_final = compile_tcprimed_files()

Processing years:   0%|          | 0/2 [00:00<?, ?it/s]/var/folders/k9/_hm_155s5md_7xqqlb895t980000gn/T/ipykernel_75079/4188285858.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_tmp], ignore_index=True)
Processing years: 100%|██████████| 2/2 [00:31<00:00, 15.77s/it]


In [5]:
df_prelim = compile_tcprimed_files(years=[2025], product='preliminary')

Processing years:   0%|          | 0/1 [00:00<?, ?it/s]/var/folders/k9/_hm_155s5md_7xqqlb895t980000gn/T/ipykernel_75079/4188285858.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_tmp], ignore_index=True)
Processing years: 100%|██████████| 1/1 [00:15<00:00, 16.00s/it]


In [6]:
df_final['product'] = ['final']*len(df_final)
df_prelim['product'] = ['preliminary']*len(df_prelim)

df = pd.concat([df_final, df_prelim], ignore_index=True)

In [8]:
df.to_csv('all-tcprimed-[2023-2025].csv', index=False)

### Match with cyclobs

In [15]:
df.head()

,year,basin,storm_number,storm_id,satellite,time,size,file_path,product
0,2023,AL,01,AL012023,AMSR2,20230115070023,0.013367,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
1,2023,AL,01,AL012023,AMSR2,20230115180333,0.013604,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
2,2023,AL,01,AL012023,AMSR2,20230116060438,0.014802,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
3,2023,AL,01,AL012023,AMSR2,20230116170836,0.015796,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
4,2023,AL,01,AL012023,AMSR2,20230117064610,0.013443,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final


In [19]:
df_cyclobs = pd.read_csv('./files/master-cyclobs-[2023-2025].csv', keep_default_na=False)

In [22]:
cyclobs_storms = df_cyclobs['storm_id'].unique()

In [24]:
df_matched = df[df['storm_id'].str.lower().isin(cyclobs_storms)]

In [34]:
df_matched['size'].sum()

np.float64(207.0544538171962)

In [27]:
df_matched.to_csv('matched-tcprimed-[2023-2025].csv', index=False)